In [1]:
import gcsfs
import pandas as pd

from _utils import GCS_FILE_PATH

In [2]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}operating_and_capital_funding.parquet",
    filesystem = gcsfs.GCSFileSystem()
)

In [3]:
df.columns

Index(['key', 'ntd_id', 'year', 'legacy_ntd_id', 'agency_status',
       'census_year', 'last_report_year', 'reporter_type', 'reporting_module',
       'uace_code', 'uza_area_sq_miles', 'primary_uza_name', 'uza_population',
       '_2024_status', 'source_agency', 'source_city', 'source_state',
       'operating_total', 'operating_federal', 'operating_state',
       'operating_local', 'operating_other', 'capital_total',
       'capital_federal', 'capital_state', 'capital_local', 'capital_other'],
      dtype='object')

In [4]:
df.source_state.unique()

array(['NV', 'CA', 'AZ', 'OR'], dtype=object)

## New Columns

In [5]:
def calculate_funding_shares(df, total_col, source_cols, suffix):
    """
    Calculate funding shares only for rows where total_col > 0.
    """

    nonzero = df[total_col] > 0

    for source, col in source_cols.items():
        share_col = f'{source}_{suffix}_share'

        df.loc[nonzero, share_col] = (
            df.loc[nonzero, col] /
            df.loc[nonzero, total_col]
        )

    return df

In [6]:
df = calculate_funding_shares(
    df,
    total_col='operating_total',
    source_cols={
        'federal': 'operating_federal',
        'state': 'operating_state',
        'local': 'operating_local',
        'other': 'operating_other'
    },
    suffix='op'
)

df = calculate_funding_shares(
    df,
    total_col='capital_total',
    source_cols={
        'federal': 'capital_federal',
        'state': 'capital_state',
        'local': 'capital_local',
        'other': 'capital_other'
    },
    suffix='cap'
)

### Federal Capital Tilt

Federal Capital Tilt measures whether an agency relies more heavily on federal funding for capital investments than for day-to-day operating expenses. A positive value means federal funding plays a larger role in capital funding than operating funding, while a negative value means the agency is more federally dependent for operations.

In [7]:
df['federal_capital_tilt'] = df['federal_cap_share'] - df['federal_op_share']

### Federal Operating Leverage

Federal Operating Leverage measures how much an agency’s total operating funding is supported by federal funding. A higher value indicates greater reliance on federal resources for ongoing operations, while a lower value indicates that the agency’s operations are funded primarily through state, local, or other sources.

In [9]:
df['federal_operating_leverage'] = (
    df['operating_federal'] / df['operating_total']
)

### Herfindahl-style concentration index (HHI)

In [7]:
op_share_cols = [
    'federal_op_share',
    'state_op_share',
    'local_op_share',
    'other_op_share'
]

df['operating_concentration'] = (
    df[op_share_cols].pow(2).sum(axis=1, min_count=1)
)

The Herfindahl-style concentration index (HHI) measures how concentrated an agency’s operating funding is across the four funding sources: federal, state, local, and other. It is calculated by squaring each source’s share of total operating funding and summing the squared shares. The index ranges from 0.25 to 1.00 when there are four funding sources: a value of 0.25 indicates that funding is evenly distributed across all four sources (25% each), while a value of 1.00 indicates that the agency receives all of its operating funding from a single source. Thus, higher HHI values indicate greater dependence on a small number of funding sources and potentially greater fiscal concentration risk, while lower values indicate a more diversified funding structure.

In [8]:
df['local_operating_funding_per_uza_resident'] = df['operating_local'] / df['uza_population']
df['operating_total_per_capita'] = df['operating_total']/ df['uza_population']

### Real Capital and Operating Funding Values using BLS Consumer Price Index Data

In [9]:
import requests
url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

payload = {
    "seriesid": ["CUUR0000SA0"],
    "startyear": "2015",
    "endyear": "2024"
}

response = requests.post(url, json=payload)
data = response.json()

In [10]:
cpi_data = data['Results']['series'][0]['data']

In [11]:
cpi = pd.DataFrame(cpi_data)

cpi['year'] = pd.to_numeric(cpi['year'])
cpi['value'] = pd.to_numeric(cpi['value'])

cpi = cpi[cpi['period'].str.startswith('M')]

cpi_annual = (
    cpi.groupby('year', as_index=False)['value']
       .mean()
       .rename(columns={'value': 'cpi'})
)

In [12]:
cpi_annual.head(5)

,year,cpi
0,2015,237.017000
1,2016,240.007167
2,2017,245.119583
3,2018,251.106833
4,2019,255.657417


In [13]:
base_cpi = cpi_annual.loc[cpi_annual['year'] == 2024, 'cpi'].iloc[0]

In [14]:
df_merged = df.merge(
    cpi_annual,
    on='year',
    how='left'
)

In [15]:
df_merged['operating_total_real'] = (
    df_merged['operating_total'] * base_cpi / df_merged['cpi']
).round(2)

df_merged['capital_total_real'] = (
    df_merged['capital_total'] * base_cpi / df_merged['cpi']
).round(2)

In [16]:
df_merged.columns

Index(['key', 'ntd_id', 'year', 'legacy_ntd_id', 'agency_status',
       'census_year', 'last_report_year', 'reporter_type', 'reporting_module',
       'uace_code', 'uza_area_sq_miles', 'primary_uza_name', 'uza_population',
       '_2024_status', 'source_agency', 'source_city', 'source_state',
       'operating_total', 'operating_federal', 'operating_state',
       'operating_local', 'operating_other', 'capital_total',
       'capital_federal', 'capital_state', 'capital_local', 'capital_other',
       'federal_op_share', 'state_op_share', 'local_op_share',
       'other_op_share', 'federal_cap_share', 'state_cap_share',
       'local_cap_share', 'other_cap_share', 'operating_concentration',
       'local_operating_funding_per_uza_resident',
       'operating_total_per_capita', 'cpi', 'operating_total_real',
       'capital_total_real'],
      dtype='object')